# PTB-XL ECG Dataset - Data Preprocessing

**Course:** AAI-501 - Introduction to AI and Machine Learning  
**Project:** ECG Arrhythmia Classification  
**Part:** 1 - Data Preparation & EDA  
**Author:** Ashok Bhairwal

## Objectives
1. Load raw ECG signals
2. Apply signal filtering (baseline removal, noise reduction)
3. Handle missing data
4. Normalize signals
5. Quality control and outlier detection
6. Save preprocessed data

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb
from scipy import signal
from scipy.signal import butter, filtfilt, medfilt
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("Libraries loaded!")

## 1. Load Dataset

In [ ]:
# Paths
DATA_PATH = Path('../data/raw/ptb-xl')
PREPROCESSED_PATH = Path('../data/preprocessed')
PREPROCESSED_PATH.mkdir(parents=True, exist_ok=True)

SAMPLING_RATE = 100  # Hz

# Load metadata
df = pd.read_csv(PREPROCESSED_PATH / 'metadata_with_labels.csv', index_col='ecg_id')
print(f"Loaded {len(df)} ECG records")
df.head()

## 2. Signal Preprocessing Functions

In [ ]:
def load_ecg_signal(ecg_id, sampling_rate=100):
    """Load ECG signal from file"""
    if sampling_rate == 100:
        path = DATA_PATH / df.loc[ecg_id, 'filename_lr']
    else:
        path = DATA_PATH / df.loc[ecg_id, 'filename_hr']
    signal, meta = wfdb.rdsamp(str(path))
    return signal

def remove_baseline_wander(signal, sampling_rate=100, cutoff=0.5):
    """Remove baseline wander using high-pass filter"""
    nyquist = sampling_rate / 2
    normal_cutoff = cutoff / nyquist
    b, a = butter(4, normal_cutoff, btype='high', analog=False)
    filtered = filtfilt(b, a, signal, axis=0)
    return filtered

def remove_powerline_interference(signal, sampling_rate=100, freq=50):
    """Remove 50/60 Hz powerline interference using notch filter"""
    nyquist = sampling_rate / 2
    freq_normalized = freq / nyquist
    b, a = signal.iirnotch(freq_normalized, Q=30)
    filtered = signal.filtfilt(b, a, signal, axis=0)
    return filtered

def bandpass_filter(signal, sampling_rate=100, lowcut=0.5, highcut=40):
    """Apply bandpass filter to keep relevant ECG frequencies"""
    nyquist = sampling_rate / 2
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(4, [low, high], btype='band')
    filtered = filtfilt(b, a, signal, axis=0)
    return filtered

def normalize_signal(signal, method='zscore'):
    """Normalize signal"""
    if method == 'zscore':
        mean = np.mean(signal, axis=0)
        std = np.std(signal, axis=0)
        normalized = (signal - mean) / (std + 1e-8)
    elif method == 'minmax':
        min_val = np.min(signal, axis=0)
        max_val = np.max(signal, axis=0)
        normalized = (signal - min_val) / (max_val - min_val + 1e-8)
    return normalized

def preprocess_ecg(ecg_id, sampling_rate=100):
    """Complete preprocessing pipeline"""
    # Load signal
    ecg_signal = load_ecg_signal(ecg_id, sampling_rate)
    
    # Apply filters
    ecg_signal = remove_baseline_wander(ecg_signal, sampling_rate)
    ecg_signal = bandpass_filter(ecg_signal, sampling_rate)
    
    # Normalize
    ecg_signal = normalize_signal(ecg_signal, method='zscore')
    
    return ecg_signal

print("Preprocessing functions defined!")

## 3. Visualize Preprocessing Steps

In [ ]:
# Select a sample ECG
sample_id = df.index[100]
raw_signal = load_ecg_signal(sample_id, SAMPLING_RATE)

# Apply preprocessing steps
signal_baseline_removed = remove_baseline_wander(raw_signal, SAMPLING_RATE)
signal_filtered = bandpass_filter(signal_baseline_removed, SAMPLING_RATE)
signal_normalized = normalize_signal(signal_filtered)

# Visualize (Lead II)
time = np.arange(raw_signal.shape[0]) / SAMPLING_RATE
lead_idx = 1  # Lead II

fig, axes = plt.subplots(4, 1, figsize=(14, 10))
fig.suptitle(f'Preprocessing Steps - ECG ID: {sample_id} (Lead II)', fontsize=14, fontweight='bold')

axes[0].plot(time, raw_signal[:, lead_idx], linewidth=0.8, color='black')
axes[0].set_ylabel('Raw Signal')
axes[0].grid(True, alpha=0.3)
axes[0].set_title('1. Raw ECG Signal')

axes[1].plot(time, signal_baseline_removed[:, lead_idx], linewidth=0.8, color='blue')
axes[1].set_ylabel('Baseline Removed')
axes[1].grid(True, alpha=0.3)
axes[1].set_title('2. After Baseline Wander Removal')

axes[2].plot(time, signal_filtered[:, lead_idx], linewidth=0.8, color='green')
axes[2].set_ylabel('Filtered')
axes[2].grid(True, alpha=0.3)
axes[2].set_title('3. After Bandpass Filter (0.5-40 Hz)')

axes[3].plot(time, signal_normalized[:, lead_idx], linewidth=0.8, color='red')
axes[3].set_ylabel('Normalized')
axes[3].set_xlabel('Time (seconds)')
axes[3].grid(True, alpha=0.3)
axes[3].set_title('4. After Z-score Normalization')

plt.tight_layout()
plt.show()

## 4. Quality Control

In [ ]:
# Filter records with quality issues
quality_cols = ['baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems']

print("Quality Control Analysis:")
print(f"Total records: {len(df)}")

# Count records with any quality issue
df['has_quality_issue'] = df[quality_cols].sum(axis=1) > 0
print(f"Records with quality issues: {df['has_quality_issue'].sum()} ({df['has_quality_issue'].sum()/len(df)*100:.1f}%)")
print(f"Clean records: {(~df['has_quality_issue']).sum()} ({(~df['has_quality_issue']).sum()/len(df)*100:.1f}%)")

# Option 1: Keep all records (handle in preprocessing)
# Option 2: Remove problematic records
print("\nRecommendation: Keep all records and handle quality issues via preprocessing")

In [ ]:
# Check for missing demographic data
print("Missing Data Check:")
key_cols = ['age', 'sex', 'NORM', 'MI', 'STTC', 'CD', 'HYP']
for col in key_cols:
    missing = df[col].isnull().sum()
    print(f"{col:15s}: {missing:5d} missing ({missing/len(df)*100:.2f}%)")

# Handle missing age/sex
df_clean = df.dropna(subset=['age', 'sex'])
print(f"\nAfter removing records with missing age/sex: {len(df_clean)} records ({len(df_clean)/len(df)*100:.1f}%)")

## 5. Process and Save Signals

In [ ]:
# Process subset of data (for demonstration)
# For full dataset, remove the .head() limitation

PROCESS_ALL = False  # Set to True to process all records
N_SAMPLES = 1000 if not PROCESS_ALL else len(df_clean)

print(f"Processing {N_SAMPLES} ECG signals...")
print("This may take several minutes...\n")

processed_signals = []
processed_ids = []
failed_ids = []

for ecg_id in tqdm(df_clean.index[:N_SAMPLES], desc="Processing"):
    try:
        # Preprocess signal
        processed_signal = preprocess_ecg(ecg_id, SAMPLING_RATE)
        processed_signals.append(processed_signal)
        processed_ids.append(ecg_id)
    except Exception as e:
        failed_ids.append(ecg_id)
        # print(f"Failed to process {ecg_id}: {e}")

print(f"\nProcessed: {len(processed_ids)} signals")
print(f"Failed: {len(failed_ids)} signals")

In [ ]:
# Convert to numpy array
X = np.array(processed_signals)  # Shape: (n_samples, n_timepoints, n_leads)
print(f"Processed signals shape: {X.shape}")
print(f"  - Number of samples: {X.shape[0]}")
print(f"  - Time points per sample: {X.shape[1]}")
print(f"  - Number of leads: {X.shape[2]}")

In [ ]:
# Get labels
superclass_labels = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
y = df_clean.loc[processed_ids, superclass_labels].values
print(f"Labels shape: {y.shape}")
print(f"\nLabel distribution:")
for i, label in enumerate(superclass_labels):
    count = y[:, i].sum()
    print(f"{label}: {count} ({count/len(y)*100:.1f}%)")

In [ ]:
# Save preprocessed data
OUTPUT_PATH = Path('../data/preprocessed')

# Save signals and labels
np.save(OUTPUT_PATH / 'X_preprocessed.npy', X)
np.save(OUTPUT_PATH / 'y_labels.npy', y)
np.save(OUTPUT_PATH / 'ecg_ids.npy', np.array(processed_ids))

# Save metadata for processed records
df_processed = df_clean.loc[processed_ids].copy()
df_processed.to_csv(OUTPUT_PATH / 'metadata_processed.csv')

print("Saved:")
print(f"  - X_preprocessed.npy: {X.shape}")
print(f"  - y_labels.npy: {y.shape}")
print(f"  - ecg_ids.npy: {len(processed_ids)}")
print(f"  - metadata_processed.csv: {df_processed.shape}")

## 6. Preprocessing Summary Statistics

In [ ]:
# Signal statistics
print("Preprocessed Signal Statistics:")
print("="*50)
print(f"Shape: {X.shape}")
print(f"Mean: {X.mean():.4f}")
print(f"Std: {X.std():.4f}")
print(f"Min: {X.min():.4f}")
print(f"Max: {X.max():.4f}")
print(f"\nPer-lead statistics (mean ± std):")
lead_names = ['I', 'II', 'III', 'AVR', 'AVL', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
for i, lead in enumerate(lead_names):
    mean = X[:, :, i].mean()
    std = X[:, :, i].std()
    print(f"{lead:3s}: {mean:6.3f} ± {std:.3f}")

In [ ]:
# Visualize distribution of preprocessed signals
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram of all values
axes[0].hist(X.flatten(), bins=100, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Signal Value')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Preprocessed Signal Values')
axes[0].set_yscale('log')

# Mean per sample
sample_means = X.mean(axis=(1, 2))
axes[1].hist(sample_means, bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('Mean Signal Value')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Sample Means')

# Std per sample
sample_stds = X.std(axis=(1, 2))
axes[2].hist(sample_stds, bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[2].set_xlabel('Std Signal Value')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Distribution of Sample Std Deviations')

plt.tight_layout()
plt.show()

## Summary

### Preprocessing Steps Applied:
1. **Baseline Wander Removal**: High-pass filter at 0.5 Hz
2. **Bandpass Filter**: 0.5-40 Hz to remove noise
3. **Normalization**: Z-score normalization per signal

### Data Quality:
- Processed signals are normalized with mean ≈ 0 and std ≈ 1
- Successfully processed majority of records
- Saved preprocessed data for feature extraction and modeling

### Output Files:
- `X_preprocessed.npy`: Preprocessed ECG signals
- `y_labels.npy`: Multi-label targets
- `ecg_ids.npy`: ECG identifiers
- `metadata_processed.csv`: Metadata for processed records

### Next Steps:
1. Time series decomposition
2. Feature extraction
3. Exploratory data analysis on processed signals